## Logistic Regression

In [ ]:
# ============================================================
# LOGISTIC REGRESSION HYPERPARAMETERS
# ============================================================

# Grid search configuration for logistic regression fine-tuning
param_grid_lr = {
    'penalty': ['l2'],  # L2 regularization (standard for logistic regression)
    'C': np.logspace(-2, 0, 7).tolist(),  # Regularization strength: 0.01, 0.0215, 0.0464, 0.1, 0.215, 0.464, 1.0
    'solver': ['lbfgs']  # Solver for L2 penalty (efficient for small datasets)
}

# Evaluation metrics for grid search
scoring = {
    'f1': make_scorer(f1_score),  # F1 score for imbalanced classification
    'auprc': make_scorer(average_precision_score),  # Area Under Precision-Recall Curve (main metric)
    'recall': make_scorer(recall_score)  # Recall (sensitivity)
}

# Cross-validation strategy (stratified to preserve class distribution)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

print("="*50)
print("1. LOGISTIC REGRESSION ON ORIGINAL FEATURES")
print("="*50)

# Grid search with cross-validation on original (raw) features
grid_search = GridSearchCV(
    estimator=LogisticRegression(max_iter=10000, class_weight='balanced', random_state=seed),
    param_grid=param_grid_lr,
    scoring=scoring,
    refit='auprc',  # Refit using AUPRC as the primary metric
    cv=skf,
    verbose=1,
    n_jobs=-1
)
grid_search.fit(finetune_data, finetune_target)

print('Best parameters:', grid_search.best_params_)
best_idx = grid_search.best_index_
print(f"Best AUPRC (CV): {grid_search.best_score_:.4f}")
print(f"F1 (CV): {grid_search.cv_results_['mean_test_f1'][best_idx]:.4f}")
print(f"Recall (CV): {grid_search.cv_results_['mean_test_recall'][best_idx]:.4f}")

# Evaluate on test set
y_pred_orig = grid_search.predict(test_data)
probs_orig = grid_search.predict_proba(test_data)[:, 1]
print("\n--- Test set ---")
print(f"AUPRC: {average_precision_score(test_target, probs_orig):.4f}")
print(f"F1: {f1_score(test_target, y_pred_orig):.4f}")
print(f"Recall: {recall_score(test_target, y_pred_orig):.4f}")
print(classification_report(test_target, y_pred_orig))